# 2. DFO-GN y métodos de comparación

**Proyecto Integrador · Métodos Numéricos · Segunda entrega**
David Castro, Alejandro Pardo, Tomás Rincón · Docente: Edwin Fernando Muñoz
Artículo base: Cartis & Roberts (2019), *A derivative-free Gauss–Newton method*.

Explica el algoritmo por partes y lo prueba en un problema.

El cuaderno es autónomo: basta con ejecutar todas las celdas en Colab o Jupyter.

In [1]:
import time, csv
from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares  # solo validación y f* de referencia

## Código de apoyo

Copia de lo explicado en el cuaderno 1, para que este cuaderno funcione sin otros archivos.

In [2]:
def eliminacion_gauss(A, B):
    """Eliminación hacia adelante con pivoteo parcial.

    A es n x n; B es n x p (p lados derechos). Con una sola eliminación se
    resuelven los p sistemas, que es la misma ventaja que tiene LU (Quiz 3, 2g).
    El multiplicador es m = a_ji / a_ii y la actualización a_jk := a_jk - m a_ik.

    Devuelve U, B modificada y razon_pivotes = min|u_ii| / max|u_ii|. Si la razón
    es casi cero, A está cerca de ser singular.
    """
    U = np.array(A, dtype=float)
    C = np.array(B, dtype=float)
    n = U.shape[0]
    for i in range(n - 1):
        # Pivoteo parcial: se sube la fila con el mayor |a_ji| para que |m| <= 1
        # y los errores de redondeo no se amplifiquen.
        p = i + int(np.argmax(np.abs(U[i:, i])))
        if U[p, i] == 0.0:
            raise np.linalg.LinAlgError("Matriz singular en la eliminación de Gauss.")
        if p != i:
            U[[i, p]] = U[[p, i]]
            C[[i, p]] = C[[p, i]]
        for j in range(i + 1, n):
            m = U[j, i] / U[i, i]
            U[j, i] = 0.0
            U[j, i + 1:] -= m * U[i, i + 1:]
            C[j] -= m * C[i]
    pivotes = np.abs(np.diag(U))
    razon = pivotes.min() / pivotes.max() if pivotes.max() > 0 else 0.0
    return U, C, razon


def sustitucion_atras(U, c):
    """Resuelve Ux = c con U triangular superior. c puede tener varias columnas."""
    n = U.shape[0]
    x = np.array(c, dtype=float)
    for i in range(n - 1, -1, -1):
        if U[i, i] == 0.0:
            raise np.linalg.LinAlgError("Pivote nulo en la sustitución hacia atrás.")
        x[i] = (x[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x


def resolver_gauss(A, B):
    """Resuelve AX = B. Devuelve X y la razón de pivotes."""
    U, C, razon = eliminacion_gauss(A, B)
    return sustitucion_atras(U, C), razon


def gram_schmidt_modificado(A, tol=1e-13):
    """QR reducida A = QR (m >= n), igual al notebook de la Semana 6.

    Cada columna v_j se actualiza justo después de cada proyección:
    v_j := v_j - r_ij q_i. tol es relativa a la mayor columna de A.
    """
    V = np.array(A, dtype=float)
    m, n = V.shape
    if m < n:
        raise ValueError("Se requiere m >= n.")
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    escala = max(np.linalg.norm(V, axis=0).max(), 1e-300)
    for i in range(n):
        R[i, i] = np.linalg.norm(V[:, i])
        if R[i, i] <= tol * escala:
            raise np.linalg.LinAlgError("Columnas linealmente dependientes.")
        Q[:, i] = V[:, i] / R[i, i]
        for j in range(i + 1, n):
            R[i, j] = Q[:, i] @ V[:, j]
            V[:, j] -= R[i, j] * Q[:, i]
    return Q, R


def minimos_cuadrados_qr(A, b):
    """Solución de min ||Ax - b||_2 mediante Rx = Q^T b (Sesión 12).

    Q^T b se calcula proyectando b de forma secuencial, con la misma idea de
    Gram-Schmidt modificado; así se conserva mejor la precisión si Q pierde
    algo de ortogonalidad.
    """
    Q, R = gram_schmidt_modificado(A)
    v = np.array(b, dtype=float)
    c = np.zeros(R.shape[0])
    for i in range(R.shape[0]):
        c[i] = Q[:, i] @ v
        v -= c[i] * Q[:, i]
    return sustitucion_atras(R, c)

In [3]:
@dataclass
class Problema:
    numero: int            # número en la Tabla 2
    nombre: str
    r: Callable            # residuos, R^n -> R^m
    J: Callable            # Jacobiano analítico, m x n
    x0: np.ndarray
    dos_f0_tabla: float
    dos_f_estrella: float

    @property
    def f_estrella(self):
        return self.dos_f_estrella / 2

    @property
    def n(self):
        return len(self.x0)

    @property
    def m(self):
        return len(self.r(self.x0))


def _rosenbrock_r(x):
    return np.array([10 * (x[1] - x[0] ** 2), 1 - x[0]])


def _rosenbrock_J(x):
    return np.array([[-20 * x[0], 10.0], [-1.0, 0.0]])


def _freudenstein_r(x):
    return np.array([-13 + x[0] + ((5 - x[1]) * x[1] - 2) * x[1],
                     -29 + x[0] + ((x[1] + 1) * x[1] - 14) * x[1]])


def _freudenstein_J(x):
    return np.array([[1.0, 10 * x[1] - 3 * x[1] ** 2 - 2],
                     [1.0, 3 * x[1] ** 2 + 2 * x[1] - 14]])


def _theta(x):
    t = np.arctan(x[1] / x[0]) / (2 * np.pi) if x[0] != 0 else 0.25
    return t + 0.5 if x[0] < 0 else t


def _helical_r(x):
    return np.array([10 * (x[2] - 10 * _theta(x)),
                     10 * (np.hypot(x[0], x[1]) - 1), x[2]])


def _helical_J(x):
    d2 = x[0] ** 2 + x[1] ** 2
    d = np.sqrt(d2)
    return np.array([[100 * x[1] / (2 * np.pi * d2), -100 * x[0] / (2 * np.pi * d2), 10.0],
                     [10 * x[0] / d, 10 * x[1] / d, 0.0],
                     [0.0, 0.0, 1.0]])


_BARD_Y = np.array([0.14, 0.18, 0.22, 0.25, 0.29, 0.32, 0.35, 0.39, 0.37, 0.58,
                    0.73, 0.96, 1.34, 2.10, 4.39])
_BARD_U = np.arange(1.0, 16.0)
_BARD_V = 16.0 - _BARD_U
_BARD_W = np.minimum(_BARD_U, _BARD_V)


def _bard_r(x):
    return _BARD_Y - (x[0] + _BARD_U / (_BARD_V * x[1] + _BARD_W * x[2]))


def _bard_J(x):
    d = (_BARD_V * x[1] + _BARD_W * x[2]) ** 2
    return np.column_stack((-np.ones(15), _BARD_U * _BARD_V / d, _BARD_U * _BARD_W / d))


def _powell_r(x):
    return np.array([x[0] + 10 * x[1], np.sqrt(5) * (x[2] - x[3]),
                     (x[1] - 2 * x[2]) ** 2, np.sqrt(10) * (x[0] - x[3]) ** 2])


def _powell_J(x):
    a = 2 * (x[1] - 2 * x[2])
    b = 2 * np.sqrt(10) * (x[0] - x[3])
    s = np.sqrt(5)
    return np.array([[1.0, 10.0, 0.0, 0.0], [0.0, 0.0, s, -s],
                     [0.0, a, -2 * a, 0.0], [b, 0.0, 0.0, -b]])


_KO_Y = np.array([0.1957, 0.1947, 0.1735, 0.1600, 0.0844, 0.0627, 0.0456,
                  0.0342, 0.0323, 0.0235, 0.0246])
_KO_U = np.array([4.0, 2.0, 1.0, 0.5, 0.25, 0.1670, 0.1250, 0.1000, 0.0833,
                  0.0714, 0.0625])


def _kowalik_r(x):
    u = _KO_U
    return _KO_Y - x[0] * (u ** 2 + u * x[1]) / (u ** 2 + u * x[2] + x[3])


def _kowalik_J(x):
    u = _KO_U
    num = u ** 2 + u * x[1]
    den = u ** 2 + u * x[2] + x[3]
    return np.column_stack((-num / den, -x[0] * u / den,
                            x[0] * num * u / den ** 2, x[0] * num / den ** 2))


def _watson_r(x):
    n = len(x)
    t = np.arange(1, 30) / 29.0
    pot = np.vander(t, n, increasing=True)            # t^(j-1), j = 1..n
    suma = pot @ x
    deriv = pot[:, :-1] @ (np.arange(1, n) * x[1:])   # suma de (j-1) x_j t^(j-2)
    return np.concatenate((deriv - suma ** 2 - 1, [x[0], x[1] - x[0] ** 2 - 1]))


def _watson_J(x):
    n = len(x)
    t = np.arange(1, 30) / 29.0
    pot = np.vander(t, n, increasing=True)
    suma = pot @ x
    J = np.zeros((31, n))
    J[:29, 0] = -2 * suma
    J[:29, 1:] = pot[:, :-1] * np.arange(1, n) - 2 * suma[:, None] * pot[:, 1:]
    J[29, 0] = 1.0
    J[30, 0] = -2 * x[0]
    J[30, 1] = 1.0
    return J


PROBLEMAS = [
    Problema(7, "Rosenbrock", _rosenbrock_r, _rosenbrock_J, np.array([-1.2, 1.0]), 24.2, 0.0),
    Problema(13, "Freudenstein & Roth", _freudenstein_r, _freudenstein_J,
             np.array([0.5, -2.0]), 400.5, 48.98425),
    Problema(9, "Helical Valley", _helical_r, _helical_J, np.array([-1.0, 0.0, 0.0]), 2500.0, 0.0),
    Problema(15, "Bard", _bard_r, _bard_J, np.array([1.0, 1.0, 1.0]), 41.68170, 8.214877e-3),
    Problema(11, "Powell Singular", _powell_r, _powell_J,
             np.array([3.0, -1.0, 0.0, 1.0]), 215.0, 0.0),
    Problema(17, "Kowalik & Osborne", _kowalik_r, _kowalik_J,
             np.array([0.25, 0.39, 0.415, 0.39]), 5.313172e-3, 3.075056e-4),
    Problema(19, "Watson", _watson_r, _watson_J, 0.5 * np.ones(6), 16.43083, 2.287670e-3),
]


class Evaluador:
    """Envuelve r(x): cuenta evaluaciones y, si se pide, agrega ruido (sección 5.2 del artículo).

    ruido = None, 'multiplicativo' (r_i (1 + e)) o 'aditivo' (r_i + e), con
    e ~ N(0, sigma^2) independiente por componente y por evaluación.
    """

    def __init__(self, problema, ruido=None, sigma=1e-2, semilla=None):
        self.problema = problema
        self.ruido = ruido
        self.sigma = sigma
        self.rng = np.random.default_rng(semilla)
        self.n_eval = 0
        self.n_eval_J = 0

    def r(self, x):
        self.n_eval += 1
        valor = self.problema.r(x)
        if self.ruido is None:
            return valor
        e = self.rng.normal(0.0, self.sigma, valor.shape)
        return valor * (1 + e) if self.ruido == "multiplicativo" else valor + e

    def J(self, x):
        self.n_eval_J += 1
        return self.problema.J(x)

    def f_real(self, x):
        """f sin ruido y sin contar: solo para medir, nunca la usa el algoritmo."""
        return 0.5 * float(np.sum(self.problema.r(x) ** 2))

## 1. Parámetros

Del Algoritmo 1 y las secciones 4.3 y 5.1 del artículo. $\Delta$ es el radio de la región de confianza: la zona donde confiamos en el modelo.

In [4]:
ETA1, ETA2 = 0.1, 0.7          # umbrales de aceptación
GAMMA_DEC, GAMMA_INC = 0.5, 2.0
DELTA_MAX = 1e10
DELTA_FIN = 1e-10
RAZON_MIN = 1e-8               # geometría: razón de pivotes mínima de W/Delta

In [5]:
def radio_inicial(x0):
    """Delta_0 = 0.1 max(||x0||_inf, 1), como en la sección 5.1 del artículo."""
    return 0.1 * max(np.linalg.norm(x0, np.inf), 1.0)

## 2. Punto de Cauchy (artículo, Supuesto 2.1)

Parte del descenso del gradiente de la S23, $x\leftarrow x-\eta\nabla f$, con $\nabla f=g=J^Tr$. En vez de escoger la tasa a mano, se usa el modelo $m(s)=f+g^Ts+\tfrac12s^THs$, con $H=J^TJ$. Sobre la dirección $-g$ el modelo es una parábola en $t$:

$$\varphi(t)=f-t\|g\|^2+\tfrac12t^2g^THg,\qquad t^*=\frac{\|g\|^2}{g^THg}.$$

Como no se puede salir de la región, $t=\min(t^*,\Delta/\|g\|)$ y $s=-tg$.

In [6]:
def paso_cauchy(g, H, delta):
    """Minimiza el modelo sobre -g dentro de la región: s = -t g.

    Es descenso del gradiente con la tasa óptima para el modelo cuadrático:
    t = g'g / g'Hg, recortada a delta / ||g||.
    """
    norma_g = np.linalg.norm(g)
    curvatura = g @ H @ g
    t_borde = delta / norma_g
    t = min(norma_g ** 2 / curvatura, t_borde) if curvatura > 0 else t_borde
    return -t * g

Comprobación con $g=(2,0)$, $H=2I$: $t^*=0.5$, de modo que $s=(-1,0)$. Con $\Delta=0.5$, el tope da $s=(-0.5,0)$.

In [7]:
print(paso_cauchy(np.array([2.0, 0.0]), 2 * np.eye(2), 10.0))
print(paso_cauchy(np.array([2.0, 0.0]), 2 * np.eye(2), 0.5))

[-1. -0.]
[-0.5 -0. ]


## 3. Paso de Gauss-Newton con QR

Linealizar da $r(x+s)\approx r+Js$. Minimizar $\|Js+r\|$ es **mínimos cuadrados** (S14) y se resuelve con QR. Si $m=n$ y $J$ es invertible, es el paso de Newton para sistemas (S18).

Se recorta a $\Delta$ y se compara con Cauchy, para respetar la condición 2.8 del artículo.

In [8]:
def valor_modelo(g, H, s):
    """Cambio del modelo m(s) - m(0) = g's + 1/2 s'Hs (ec. 2.5)."""
    return g @ s + 0.5 * s @ H @ s

In [9]:
def paso_gn_qr(J, r, g, H, delta):
    """Paso de Gauss-Newton: min ||J s + r|| por QR, recortado al radio delta.

    Si J está casi singular, el paso recortado puede bajar el modelo menos que el
    de Cauchy e incumplir la condición 2.8 del artículo, que sostiene la
    convergencia global. Por eso se devuelve el mejor de los dos según el modelo.
    """
    cauchy = paso_cauchy(g, H, delta)
    try:
        s = minimos_cuadrados_qr(J, -r)
    except np.linalg.LinAlgError:
        return cauchy
    norma = np.linalg.norm(s)
    if norma > delta:
        s = s * (delta / norma)
    return s if valor_modelo(g, H, s) <= valor_modelo(g, H, cauchy) else cauchy

## 4. Tres formas de obtener $J$

- **Exacto:** referencia con derivadas.
- **Diferencias finitas (Unidad 2):** $n$ evaluaciones nuevas, con $h\approx\sqrt{\epsilon}$ (Unidad 1).

In [10]:
class JacobianoExacto:
    def __init__(self, ev, x, rx, delta):
        self.ev = ev
        self.J = ev.J(x)

    def tras_paso(self, x_nuevo, r_nuevo, aceptado, x, rx, delta):
        if aceptado:
            self.J = self.ev.J(x_nuevo)

In [11]:
class JacobianoDiferencias:
    """Diferencias finitas hacia adelante (Unidad 2): n evaluaciones por Jacobiano."""
    H_BASE = np.sqrt(np.finfo(float).eps)   # paso óptimo aprox. sqrt(épsilon), Unidad 1

    def __init__(self, ev, x, rx, delta):
        self.ev = ev
        self.J = self._calcular(x, rx)

    def _calcular(self, x, rx):
        J = np.zeros((len(rx), len(x)))
        for j in range(len(x)):
            h = self.H_BASE * max(1.0, abs(x[j]))
            y = x.copy()
            y[j] += h
            J[:, j] = (self.ev.r(y) - rx) / h
        return J

    def tras_paso(self, x_nuevo, r_nuevo, aceptado, x, rx, delta):
        if aceptado:
            self.J = self._calcular(x_nuevo, r_nuevo)

- **Interpolación (la idea central del artículo, ec. 2.2–2.4):** se exige que el modelo lineal pase por $n+1$ puntos. Con $W$ de filas $(y_t-x)^T$ y $D$ de filas $r(y_t)-r(x)$, queda

$$WJ^T=D,$$

que se resuelve con **una** eliminación de Gauss. Si los puntos se alinean, la razón de pivotes cae y se reconstruyen. Esta regla es nuestra simplificación de la geometría del artículo.

In [12]:
class JacobianoInterpolacion:
    """Jacobiano de DFO-GN: interpolación lineal con n+1 puntos (ec. 2.2 a 2.4).

    Por cada residuo r_i se pide m_i(y_t - x) = r_i(y_t). Con W de filas (y_t - x)'
    y D de filas r(y_t) - r(x), eso equivale a W J' = D, que se resuelve con una
    sola eliminación de Gauss para los m residuos.
    """

    def __init__(self, ev, x, rx, delta):
        self.ev = ev
        self._reconstruir(x, delta)
        self._ajustar(x, rx, delta)

    def _reconstruir(self, x, delta):
        """Y = {x + delta e_j}: base canónica, la geometría ideal. Cuesta n evaluaciones."""
        self.Y = [x + delta * e for e in np.eye(len(x))]
        self.rY = [self.ev.r(y) for y in self.Y]

    def _sistema(self, x, rx, delta):
        W = np.array([y - x for y in self.Y]) / delta   # se escala para medir solo la forma
        D = np.array([ry - rx for ry in self.rY]) / delta
        return resolver_gauss(W, D)

    def _ajustar(self, x, rx, delta):
        try:
            Jt, razon = self._sistema(x, rx, delta)
        except np.linalg.LinAlgError:
            razon = 0.0
        if razon < RAZON_MIN:              # puntos casi alineados: geometría degenerada
            self._reconstruir(x, delta)
            Jt, razon = self._sistema(x, rx, delta)
        self.J = Jt.T

    def tras_paso(self, x_nuevo, r_nuevo, aceptado, x, rx, delta):
        """Entra el punto de prueba y sale el más lejano del centro de la región."""
        centro, r_centro = (x_nuevo, r_nuevo) if aceptado else (x, rx)
        puntos = self.Y + [x if aceptado else x_nuevo]
        valores = self.rY + [rx if aceptado else r_nuevo]
        distancias = [np.linalg.norm(p - centro) for p in puntos]
        sale = int(np.argmax(distancias))
        del puntos[sale], valores[sale]
        self.Y, self.rY = puntos, valores
        # Tras un rechazo, si los puntos quedaron lejos el modelo ya no representa
        # la región; se reconstruye en vez de seguir encogiendo el radio.
        if not aceptado and max(np.linalg.norm(p - centro) for p in self.Y) > 2 * delta:
            self._reconstruir(centro, delta)
        self._ajustar(centro, r_centro, delta)

## 5. Núcleo: región de confianza (artículo, sección 2.2)

$$R_k=\frac{f(x_k)-f(x_k+s_k)}{m_k(0)-m_k(s_k)}$$

Si $R_k\ge0.7$ se acepta y $\Delta$ crece; si $0.1\le R_k<0.7$ se acepta y $\Delta$ se reduce; si $R_k<0.1$ se rechaza y $\Delta$ se reduce.

In [13]:
def region_confianza(ev, tipo_J, tipo_paso, presupuesto=None):
    """Región de confianza con cociente R_k (ec. 2.6 y 2.7, regla simplificada 2.15)."""
    inicio = time.perf_counter()
    x = ev.problema.x0.astype(float).copy()
    n = len(x)
    presupuesto = presupuesto or 200 * (n + 1)
    delta = radio_inicial(x)
    rx = ev.r(x)
    fx = 0.5 * rx @ rx
    f0 = fx
    clases = {"interpolacion": JacobianoInterpolacion, "exacto": JacobianoExacto,
              "diferencias": JacobianoDiferencias}
    modelo = clases[tipo_J](ev, x, rx, delta)
    calcular_paso = paso_cauchy if tipo_paso == "cauchy" else None
    historial = [(ev.n_eval, ev.f_real(x), 0)]
    iterados = [x.copy()]
    motivo = "presupuesto"
    k = 0
    while ev.n_eval < presupuesto:
        k += 1
        J = modelo.J
        g = J.T @ rx
        H = J.T @ J
        if fx <= max(1e-12, 1e-20 * f0):
            motivo = "f pequeña"
            break
        if delta < DELTA_FIN:
            motivo = "radio pequeño"
            break
        if np.linalg.norm(g) == 0.0:
            motivo = "gradiente nulo"
            break
        s = calcular_paso(g, H, delta) if calcular_paso else paso_gn_qr(J, rx, g, H, delta)
        x_nuevo = x + s
        r_nuevo = ev.r(x_nuevo)
        f_nuevo = 0.5 * r_nuevo @ r_nuevo
        predicha = -valor_modelo(g, H, s)
        R = (fx - f_nuevo) / predicha if predicha > 0 else -1.0
        if not np.isfinite(R):
            R = -1.0
        aceptado = R >= ETA1
        norma_s = np.linalg.norm(s)
        if R >= ETA2:
            delta = min(max(GAMMA_INC * delta, GAMMA_INC * norma_s), DELTA_MAX)
        elif aceptado:
            delta = max(GAMMA_DEC * delta, norma_s)
        else:
            delta = min(GAMMA_DEC * delta, norma_s)
        modelo.tras_paso(x_nuevo, r_nuevo, aceptado, x, rx, delta)
        if aceptado:
            x, rx, fx = x_nuevo, r_nuevo, f_nuevo
            iterados.append(x.copy())
        historial.append((ev.n_eval, ev.f_real(x), k))
    return {"x": x, "f": ev.f_real(x), "historial": historial, "n_eval": ev.n_eval,
            "n_eval_J": ev.n_eval_J, "n_iter": k, "motivo": motivo, "iterados": iterados,
            "radio_final": delta, "tiempo": time.perf_counter() - inicio}

## 6. Descenso del gradiente (S23)

Comparador con $\eta$ fija y $J$ exacto.

In [14]:
def descenso_gradiente(ev, eta, presupuesto=None):
    """x := x - eta grad f, con grad f = J'r y tasa fija (Sesión 23)."""
    inicio = time.perf_counter()
    x = ev.problema.x0.astype(float).copy()
    presupuesto = presupuesto or 200 * (len(x) + 1)
    historial = [(0, ev.f_real(x), 0)]
    motivo = "presupuesto"
    k = 0
    while ev.n_eval < presupuesto:
        k += 1
        g = ev.J(x).T @ ev.r(x)
        if not np.isfinite(g).all():
            motivo = "divergió"
            break
        x = x - eta * g
        with np.errstate(all="ignore"):
            f = ev.f_real(x)
        if not np.isfinite(f):
            motivo = "divergió"
            historial.append((ev.n_eval, np.inf, k))
            break
        historial.append((ev.n_eval, f, k))
    return {"x": x, "f": historial[-1][1], "historial": historial, "n_eval": ev.n_eval,
            "n_eval_J": ev.n_eval_J, "n_iter": k, "motivo": motivo,
            "tiempo": time.perf_counter() - inicio}

## 7. Prueba en Rosenbrock (problema 7 de la Tabla 2)

In [15]:
p = PROBLEMAS[0]
filas = []
for nombre, (tipo_J, paso) in {"DFO-GN Cauchy": ("interpolacion", "cauchy"), "DFO-GN QR": ("interpolacion", "qr"),
                               "GN exacto QR": ("exacto", "qr"), "GN dif. finitas QR": ("diferencias", "qr")}.items():
    res = region_confianza(Evaluador(p), tipo_J, paso)
    filas.append([nombre, res["f"], res["n_eval"], res["n_eval_J"], res["n_iter"], res["motivo"]])
filas.append(["Descenso gradiente (eta=0.01)", (d := descenso_gradiente(Evaluador(p), 1e-2))["f"], d["n_eval"], d["n_eval_J"], d["n_iter"], d["motivo"]])
pd.DataFrame(filas, columns=["método", "f final", "evals r", "evals J", "iteraciones", "parada"])

,método,f final,evals r,evals J,iteraciones,parada
0,DFO-GN Cauchy,0.002047,600,0,597,presupuesto
1,DFO-GN QR,0.000000,75,0,57,f pequeña
2,GN exacto QR,0.000000,29,21,29,f pequeña
3,GN dif. finitas QR,0.000000,71,0,29,f pequeña
4,Descenso gradiente (eta=0.01),0.158996,600,600,600,presupuesto


**Lectura:** con QR se llega a $f^*=0$ dentro del presupuesto, tanto con $J$ exacto como sin derivadas. Con Cauchy no. El análisis completo está en el cuaderno 3.